In [0]:
%run /Shared/insclm_capstone/NB_00_config_loader.py

[SecretScope(name=' kv-insclm-cap-11'), SecretScope(name='kv-insclm')]

[SecretMetadata(key='adls-abfss-base'),
 SecretMetadata(key='adls-account-key'),
 SecretMetadata(key='adls-account-name'),
 SecretMetadata(key='adls-audit-path'),
 SecretMetadata(key='adls-base-url'),
 SecretMetadata(key='adls-bronze-path'),
 SecretMetadata(key='adls-container-name'),
 SecretMetadata(key='adls-gold-path'),
 SecretMetadata(key='adls-raw-path'),
 SecretMetadata(key='adls-rejected-path'),
 SecretMetadata(key='adls-silver-path'),
 SecretMetadata(key='database-workspace-url'),
 SecretMetadata(key='databricks-cluster-id'),
 SecretMetadata(key='databricks-pat'),
 SecretMetadata(key='file-claim-status-updates'),
 SecretMetadata(key='file-claims'),
 SecretMetadata(key='file-customer-master'),
 SecretMetadata(key='file-policy-master'),
 SecretMetadata(key='github-pat'),
 SecretMetadata(key='github-repo-url'),
 SecretMetadata(key='sql-admin-name'),
 SecretMetadata(key='sql-admin-password'),
 SecretMetadata(key='sql-connection-string'),
 SecretMetadata(key='sql-database-name'),
 S

✅ Config loaded from Key Vault successfully.
   ADLS Account  : [REDACTED]
   Container     : [REDACTED]
   ABFSS Base    : [REDACTED]
   RAW path      : [REDACTED][REDACTED]
   BRONZE path   : [REDACTED][REDACTED]
   SILVER path   : [REDACTED][REDACTED]
   GOLD path     : [REDACTED][REDACTED]
   REJECTED path : [REDACTED][REDACTED]
   AUDIT path    : [REDACTED][REDACTED]
   SQL Server    : [REDACTED]
   SQL Database  : [REDACTED]


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, TimestampType)
from datetime import datetime

print("=" * 55)
print("AUDIT PIPELINE LOG")
print("=" * 55)

AUDIT PIPELINE LOG


In [0]:
# Create audit database
# Unity Catalog — NO LOCATION
spark.sql("CREATE DATABASE IF NOT EXISTS audit_insclm")
print("✅ audit_insclm database ready")

audit_schema = StructType([
    StructField("run_id",           StringType(),    False),
    StructField("pipeline_name",    StringType(),    False),
    StructField("source_name",      StringType(),    True),
    StructField("target_name",      StringType(),    True),
    StructField("load_type",        StringType(),    True),
    StructField("records_read",     LongType(),      True),
    StructField("records_inserted", LongType(),      True),
    StructField("records_updated",  LongType(),      True),
    StructField("records_rejected", LongType(),      True),
    StructField("status",           StringType(),    True),
    StructField("error_message",    StringType(),    True),
    StructField("notes",            StringType(),    True),
    StructField("run_ts",           TimestampType(), True),
])

TABLE_NAME = "audit_insclm.audit_pipeline_log"

if not spark.catalog.tableExists(TABLE_NAME):
    (spark.createDataFrame([], audit_schema)
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TABLE_NAME))
    print("✅ audit_pipeline_log table created")
else:
    print("✅ audit_pipeline_log already exists")

✅ audit_insclm database ready
✅ audit_pipeline_log table created


In [0]:
# Helper function to log each stage
def log_audit(run_id, pipeline_name, source_name,
              target_name, load_type,
              records_read=0, records_inserted=0,
              records_updated=0, records_rejected=0,
              status="SUCCESS", error_message=None,
              notes=None):

    row = [(
        run_id,
        pipeline_name,
        source_name,
        target_name,
        load_type,
        records_read,
        records_inserted,
        records_updated,
        records_rejected,
        status,
        error_message,
        notes,
        datetime.now()
    )]

    df = spark.createDataFrame(row, audit_schema)
    df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(TABLE_NAME)

print("✅ log_audit function ready")

✅ log_audit function ready


In [0]:
# Get actual row counts from all tables
print("\n📥 Reading row counts from all tables...")

def safe_count(table_name):
    try:
        return spark.table(table_name).count()
    except:
        return 0

counts = {
    "bronze_claims":
        safe_count("bronze_insclm.bronze_claims"),
    "bronze_claim_status_updates":
        safe_count(
            "bronze_insclm.bronze_claim_status_updates"),
    "bronze_policy_master":
        safe_count("bronze_insclm.bronze_policy_master"),
    "bronze_customer_master":
        safe_count("bronze_insclm.bronze_customer_master"),
    "silver_customer_dim":
        safe_count("silver_insclm.silver_customer_dim"),
    "silver_claims_fact":
        safe_count("silver_insclm.silver_claims_fact"),
    "silver_policy_dim":
        safe_count("silver_insclm.silver_policy_dim"),
    "silver_claim_status_history":
        safe_count(
            "silver_insclm.silver_claim_status_history"),
    "rejected_claims":
        safe_count("rejected_insclm.rejected_claims"),
    "rejected_status_updates":
        safe_count(
            "rejected_insclm.rejected_status_updates"),
    "rejected_policy":
        safe_count("rejected_insclm.rejected_policy"),
    "rejected_customers":
        safe_count("rejected_insclm.rejected_customers"),
    "gold_claim_summary":
        safe_count("gold_insclm.gold_claim_summary"),
    "gold_policy_history_summary":
        safe_count(
            "gold_insclm.gold_policy_history_summary"),
    "gold_suspicious_claim_summary":
        safe_count(
            "gold_insclm.gold_suspicious_claim_summary"),
}

for table, count in counts.items():
    print(f"   {table:<40}: {count:,}")


📥 Reading row counts from all tables...
   bronze_claims                           : 2,200
   bronze_claim_status_updates             : 1,600
   bronze_policy_master                    : 1,500
   bronze_customer_master                  : 1,000
   silver_customer_dim                     : 1,000
   silver_claims_fact                      : 2,080
   silver_policy_dim                       : 1,500
   silver_claim_status_history             : 1,600
   rejected_claims                         : 120
   rejected_status_updates                 : 0
   rejected_policy                         : 0
   rejected_customers                      : 0
   gold_claim_summary                      : 30
   gold_policy_history_summary             : 1,500
   gold_suspicious_claim_summary           : 1,273


In [0]:
# Log all pipeline stages
print("\n📥 Logging all pipeline stages...")

RUN_ID = f"CAPSTONE06_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(f"   Batch Run ID: {RUN_ID}\n")

audit_entries = [
    # NB_01 Bronze ingestion
    ("NB_01_bronze_ingestion",
     "raw/claims/claims.csv",
     "bronze_insclm.bronze_claims",
     "FULL_LOAD",
     counts["bronze_claims"],
     counts["bronze_claims"],
     0, 0, "SUCCESS", None,
     "Initial ingestion from claims.csv"),

    ("NB_01_bronze_ingestion",
     "raw/claim_status_updates/",
     "bronze_insclm.bronze_claim_status_updates",
     "FULL_LOAD",
     counts["bronze_claim_status_updates"],
     counts["bronze_claim_status_updates"],
     0, 0, "SUCCESS", None,
     "Initial ingestion from claim_status_updates.csv"),

    ("NB_01_bronze_ingestion",
     "raw/policy_master/",
     "bronze_insclm.bronze_policy_master",
     "FULL_LOAD",
     counts["bronze_policy_master"],
     counts["bronze_policy_master"],
     0, 0, "SUCCESS", None,
     "Ingested from ADF dated folder"),

    ("NB_01_bronze_ingestion",
     "raw/customer_master/",
     "bronze_insclm.bronze_customer_master",
     "FULL_LOAD",
     counts["bronze_customer_master"],
     counts["bronze_customer_master"],
     0, 0, "SUCCESS", None,
     "Ingested from ADF dated folder"),

    # NB_02 Silver transformation
    ("NB_02_silver_transformation",
     "bronze_insclm.bronze_customer_master",
     "silver_insclm.silver_customer_dim",
     "FULL_LOAD",
     counts["bronze_customer_master"],
     counts["silver_customer_dim"],
     0, 0, "SUCCESS", None,
     "Cleaned + age_years + email_valid + phone_valid"),

    ("NB_02_silver_transformation",
     "bronze_insclm.bronze_claims",
     "rejected_insclm.rejected_claims",
     "FULL_LOAD",
     counts["bronze_claims"],
     counts["rejected_claims"],
     0, counts["rejected_claims"],
     "SUCCESS", None,
     "Rejected: MISSING_DOCUMENT_STATUS + others"),

    ("NB_02_silver_transformation",
     "bronze + policy_dim + customer_dim",
     "silver_insclm.silver_claims_fact",
     "FULL_LOAD",
     counts["bronze_claims"] - counts["rejected_claims"],
     counts["silver_claims_fact"],
     0, counts["rejected_claims"],
     "SUCCESS", None,
     "3-way join + 4 business flags + suspicious_score"),

    # NB_03 Rejected records
    ("NB_03_silver_rejected_records",
     "bronze_insclm.bronze_claim_status_updates",
     "rejected_insclm.rejected_status_updates",
     "FULL_LOAD",
     counts["bronze_claim_status_updates"],
     counts["rejected_status_updates"],
     0, counts["rejected_status_updates"],
     "SUCCESS", None,
     "0 rejections — data is clean"),

    ("NB_03_silver_rejected_records",
     "bronze_insclm.bronze_policy_master",
     "rejected_insclm.rejected_policy",
     "FULL_LOAD",
     counts["bronze_policy_master"],
     counts["rejected_policy"],
     0, counts["rejected_policy"],
     "SUCCESS", None,
     "0 rejections — data is clean"),

    ("NB_03_silver_rejected_records",
     "bronze_insclm.bronze_customer_master",
     "rejected_insclm.rejected_customers",
     "FULL_LOAD",
     counts["bronze_customer_master"],
     0, 0, 0,
     "SUCCESS", None,
     "0 rejections — dob NULL issue resolved"),

    # NB_04 SCD Type 2
    ("NB_04_scd_policy_dim",
     "bronze_insclm.bronze_policy_master",
     "silver_insclm.silver_policy_dim",
     "SCD_TYPE2",
     counts["bronze_policy_master"],
     counts["silver_policy_dim"],
     0, 0, "SUCCESS", None,
     "SCD Type 2 — effective/expiry/is_current/hash"),

    # NB_05 Delta MERGE
    ("NB_05_merge_claim_status",
     "bronze_insclm.bronze_claim_status_updates",
     "silver_insclm.silver_claim_status_history",
     "DELTA_MERGE",
     counts["bronze_claim_status_updates"],
     counts["silver_claim_status_history"],
     0, 0, "SUCCESS", None,
     "Delta MERGE upsert on status_update_id"),

    # NB_06 Gold tables
    ("NB_06_gold_tables",
     "silver_insclm.silver_claims_fact",
     "gold_insclm.gold_claim_summary",
     "AGGREGATION",
     counts["silver_claims_fact"],
     counts["gold_claim_summary"],
     0, 0, "SUCCESS", None,
     "Grouped by policy_type + claim_reason"),

    ("NB_06_gold_tables",
     "silver_insclm.silver_policy_dim",
     "gold_insclm.gold_policy_history_summary",
     "AGGREGATION",
     counts["silver_policy_dim"],
     counts["gold_policy_history_summary"],
     0, 0, "SUCCESS", None,
     "Policy version history + coverage_change_pct"),

    ("NB_06_gold_tables",
     "silver_insclm.silver_claims_fact",
     "gold_insclm.gold_suspicious_claim_summary",
     "FILTER",
     counts["silver_claims_fact"],
     counts["gold_suspicious_claim_summary"],
     0, 0, "SUCCESS", None,
     "Filtered suspicious_score >= 1"),

    # NB_09 Azure SQL load
    ("NB_09_load_to_azure_sql",
     "gold_insclm.gold_claim_summary",
     "reporting.fact_claim_summary",
     "JDBC_WRITE",
     counts["gold_claim_summary"],
     counts["gold_claim_summary"],
     0, 0, "SUCCESS", None,
     "Written to Azure SQL via JDBC"),

    ("NB_09_load_to_azure_sql",
     "gold_insclm.gold_suspicious_claim_summary",
     "reporting.fact_suspicious_claims",
     "JDBC_WRITE",
     counts["gold_suspicious_claim_summary"],
     counts["gold_suspicious_claim_summary"],
     0, 0, "SUCCESS", None,
     "Written to Azure SQL via JDBC"),

    ("NB_09_load_to_azure_sql",
     "silver_insclm.silver_policy_dim",
     "reporting.dim_policy_history",
     "JDBC_WRITE",
     counts["silver_policy_dim"],
     counts["silver_policy_dim"],
     0, 0, "SUCCESS", None,
     "Written to Azure SQL via JDBC"),
]

for entry in audit_entries:
    log_audit(
        run_id           = RUN_ID,
        pipeline_name    = entry[0],
        source_name      = entry[1],
        target_name      = entry[2],
        load_type        = entry[3],
        records_read     = entry[4],
        records_inserted = entry[5],
        records_updated  = entry[6],
        records_rejected = entry[7],
        status           = entry[8],
        error_message    = entry[9],
        notes            = entry[10]
    )
    print(f"✅ Logged: {entry[2]}")


📥 Logging all pipeline stages...
   Batch Run ID: CAPSTONE06_20260521_154048

✅ Logged: bronze_insclm.bronze_claims
✅ Logged: bronze_insclm.bronze_claim_status_updates
✅ Logged: bronze_insclm.bronze_policy_master
✅ Logged: bronze_insclm.bronze_customer_master
✅ Logged: silver_insclm.silver_customer_dim
✅ Logged: rejected_insclm.rejected_claims
✅ Logged: silver_insclm.silver_claims_fact
✅ Logged: rejected_insclm.rejected_status_updates
✅ Logged: rejected_insclm.rejected_policy
✅ Logged: rejected_insclm.rejected_customers
✅ Logged: silver_insclm.silver_policy_dim
✅ Logged: silver_insclm.silver_claim_status_history
✅ Logged: gold_insclm.gold_claim_summary
✅ Logged: gold_insclm.gold_policy_history_summary
✅ Logged: gold_insclm.gold_suspicious_claim_summary
✅ Logged: reporting.fact_claim_summary
✅ Logged: reporting.fact_suspicious_claims
✅ Logged: reporting.dim_policy_history


In [0]:
# Show full audit log
print("\n" + "=" * 55)
print("FULL AUDIT LOG")
print("=" * 55)

spark.table(TABLE_NAME) \
    .filter(F.col("run_id") == RUN_ID) \
    .select(
        "pipeline_name",
        "source_name",
        "target_name",
        "load_type",
        "records_read",
        "records_inserted",
        "records_rejected",
        "status"
    ).show(20, truncate=False)


FULL AUDIT LOG
+-----------------------------+-----------------------------------------+-----------------------------------------+-----------+------------+----------------+----------------+-------+
|pipeline_name                |source_name                              |target_name                              |load_type  |records_read|records_inserted|records_rejected|status |
+-----------------------------+-----------------------------------------+-----------------------------------------+-----------+------------+----------------+----------------+-------+
|NB_05_merge_claim_status     |bronze_insclm.bronze_claim_status_updates|silver_insclm.silver_claim_status_history|DELTA_MERGE|1600        |1600            |0               |SUCCESS|
|NB_02_silver_transformation  |bronze_insclm.bronze_customer_master     |silver_insclm.silver_customer_dim        |FULL_LOAD  |1000        |1000            |0               |SUCCESS|
|NB_02_silver_transformation  |bronze + policy_dim + customer_dim    

In [0]:
# Final summary
total_logged   = spark.table(TABLE_NAME) \
    .filter(F.col("run_id") == RUN_ID).count()
total_read     = spark.table(TABLE_NAME) \
    .filter(F.col("run_id") == RUN_ID) \
    .agg(F.sum("records_read")).first()[0] or 0
total_inserted = spark.table(TABLE_NAME) \
    .filter(F.col("run_id") == RUN_ID) \
    .agg(F.sum("records_inserted")).first()[0] or 0
total_rejected = spark.table(TABLE_NAME) \
    .filter(F.col("run_id") == RUN_ID) \
    .agg(F.sum("records_rejected")).first()[0] or 0

print("\n" + "=" * 55)
print("AUDIT SUMMARY")
print("=" * 55)
print(f"Run ID              : {RUN_ID}")
print(f"Total stages logged : {total_logged:,}")
print(f"Total records read  : {total_read:,}")
print(f"Total records saved : {total_inserted:,}")
print(f"Total records reject: {total_rejected:,}")
print("=" * 55)
print("✅ NB_10 complete.")
print("✅ ALL 10 NOTEBOOKS DONE!")
print("\nFinal steps:")
print("1. Export all notebooks as .py from Databricks")
print("2. Commit to GitHub")
print("3. Take screenshots of all outputs")
print("=" * 55)


AUDIT SUMMARY
Run ID              : CAPSTONE06_20260521_154048
Total stages logged : 18
Total records read  : 27,243
Total records saved : 18,206
Total records reject: 240
✅ NB_10 complete.
✅ ALL 10 NOTEBOOKS DONE!

Final steps:
1. Export all notebooks as .py from Databricks
2. Commit to GitHub
3. Take screenshots of all outputs
